# LAB S03: OCR y procesamiento de imágenes

Cada sección usa el archivo de `mi_pipeline` que se escribe en su paso de la guía; el cuaderno solo importa y ejecuta. Se trabaja con documentos ficticios (y, si existe, con el lote público real en las secciones opcionales); no se suben a servicios en la nube.

## 0. Dónde están los datos de práctica

In [ ]:
# Celda estándar de localización de datos (no se modifica)
import os
from pathlib import Path


def localizar_carpeta(nombre="02_Datos_Practica", max_niveles=6, obligatorio=True):
    """Busca `nombre` en la carpeta actual y en sus carpetas superiores."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        candidata = carpeta / nombre
        if candidata.is_dir():
            return candidata
    # Respaldo: carpetas del curso en la unidad del sistema (VM de laboratorio)
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    respaldo = [curso / "paquete" / nombre]
    if nombre == "02_Datos_Practica":
        respaldo.insert(0, curso / "datos")
    for candidata in respaldo:
        if candidata.is_dir():
            return candidata
    if obligatorio:
        raise FileNotFoundError(
            f"No se encuentra la carpeta '{nombre}'. El cuaderno se abre desde su carpeta "
            "dentro del repositorio del curso, sin moverlo de lugar."
        )
    return None


DATOS = localizar_carpeta("02_Datos_Practica")
print("Datos de práctica encontrados:", DATOS.name)
print("Subcarpetas:", sorted(p.name for p in DATOS.iterdir() if p.is_dir()))


## Paso 1: Conexión con `mi_pipeline` y control C0

In [ ]:
# Conexión de mi_pipeline y de las herramientas del LAB (no se modifica)
%load_ext autoreload
%autoreload 2
import json
import re
import shutil
import subprocess
import sys
import time

import cv2
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)


def localizar_raiz(max_niveles=6):
    """Carpeta del repositorio clonado que contiene mi_pipeline."""
    if os.environ.get("MI_PIPELINE_RAIZ"):
        return Path(os.environ["MI_PIPELINE_RAIZ"])
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        if (carpeta / "mi_pipeline" / "__init__.py").is_file():
            return carpeta
    clon = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso" / "trabajo" / "entorno-extraccion-documental"
    if (clon / "mi_pipeline" / "__init__.py").is_file():
        return clon
    raise FileNotFoundError("No se encuentra la carpeta mi_pipeline: se crea en el paso «Crear la carpeta mi_pipeline» de la guía.")


RAIZ = localizar_raiz()
sys.path.insert(0, str(RAIZ))
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)
CONTROLES = []
INICIO = time.perf_counter()
pytesseract.pytesseract.tesseract_cmd = (
    os.environ.get("TESSERACT_CMD")
    or shutil.which("tesseract")
    or str(Path(os.environ.get("ProgramFiles", "")) / "Tesseract-OCR" / "tesseract.exe")
)


def correr_pruebas():
    """Corre las pruebas de mi_pipeline con pytest, muestra el resumen y dice si todas pasaron."""
    corrida = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(RAIZ / "mi_pipeline" / "tests")],
        capture_output=True, text=True, cwd=str(RAIZ), timeout=300, encoding="utf-8", errors="replace",
    )
    resumen = [linea for linea in corrida.stdout.strip().splitlines() if " passed" in linea or " failed" in linea or "error" in linea]
    print(re.sub(r" in [\d.]+s.*", "", resumen[-1]) if resumen else corrida.stdout[-500:])
    return corrida.returncode == 0


print("Tesseract:", pytesseract.get_tesseract_version())
print("OpenCV:", cv2.__version__, "| carpeta del repositorio:", RAIZ.name)


In [ ]:
# ✔ Control C0
import mi_pipeline

assert mi_pipeline.__name__ == "mi_pipeline"
CONTROLES.append("C0")
print("✔ C0 superado: mi_pipeline se importa sin error, versión", mi_pipeline.__version__)

In [ ]:
# Herramientas para ver los documentos antes y después de cada paso (no se modifica)
import unicodedata

from IPython.display import display
from PIL import ImageDraw, ImageFont


def cargar_fuente(tamano=18):
    """Una tipografía del sistema con tildes; si no hay ninguna, la tipografía por defecto (sin tildes)."""
    for nombre in ("arial.ttf", "segoeui.ttf", "DejaVuSans.ttf"):
        try:
            return ImageFont.truetype(nombre, tamano), True
        except OSError:
            continue
    return ImageFont.load_default(), False


FUENTE, CON_TILDES = cargar_fuente()


def a_pil(imagen):
    """Convierte una matriz de NumPy (gris, binaria o color) en una imagen de Pillow a color."""
    if isinstance(imagen, Image.Image):
        return imagen.convert("RGB")
    matriz = np.asarray(imagen)
    if matriz.dtype == bool:
        matriz = matriz.astype("uint8") * 255
    return Image.fromarray(matriz).convert("RGB")


def recortar(imagen, y=(0.0, 1.0), x=(0.0, 1.0)):
    """Recorta una región con fracciones de la altura (y) y del ancho (x): (0.0, 1.0) es todo."""
    pil = a_pil(imagen)
    ancho, alto = pil.size
    return pil.crop((int(ancho * x[0]), int(alto * y[0]), int(ancho * x[1]), int(alto * y[1])))


def con_cajas(imagen, cajas, colores, grosor=4):
    """Dibuja rectángulos (x1, y1, x2, y2) de colores sobre una copia de la imagen."""
    lienzo = np.array(a_pil(imagen))
    for caja, color in zip(cajas, colores):
        cv2.rectangle(lienzo, (int(caja[0]), int(caja[1])), (int(caja[2]), int(caja[3])), color, grosor)
    return lienzo


def ver(imagenes, titulos, ancho=420, por_fila=3, exacto=False):
    """Muestra varias imágenes en cuadrícula, cada una con su título y todas con el mismo ancho en pantalla.

    Con exacto=True se agranda sin suavizar, para ver los puntos (píxeles) de cada imagen."""
    fichas = []
    for imagen, titulo in zip(imagenes, titulos):
        if not CON_TILDES:
            titulo = unicodedata.normalize("NFKD", titulo).encode("ascii", "ignore").decode()
        pil = a_pil(imagen)
        pil = pil.resize((ancho, max(1, round(pil.height * ancho / pil.width))), Image.NEAREST if exacto else Image.BICUBIC)
        ficha = Image.new("RGB", (ancho, pil.height + 30), "white")
        ficha.paste(pil, (0, 30))
        ImageDraw.Draw(ficha).text((4, 4), titulo, fill="black", font=FUENTE)
        fichas.append(ficha)
    filas = [fichas[i:i + por_fila] for i in range(0, len(fichas), por_fila)]
    lienzo = Image.new("RGB", (por_fila * ancho + 8 * (por_fila - 1), sum(max(f.height for f in fila) + 8 for fila in filas) - 8), "#B8BCCB")
    y = 0
    for fila in filas:
        for i, ficha in enumerate(fila):
            lienzo.paste(ficha, (i * (ancho + 8), y))
        y += max(f.height for f in fila) + 8
    display(lienzo)

## Reto: ¿Por qué un comprobante fotografiado sale ilegible?

In [ ]:
# Reto: el comprobante fotografiado, leído por Tesseract sin ningún tratamiento
vou = DATOS / "08_Vouchers" / "VOU-0001.jpg"
texto_reto = pytesseract.image_to_string(Image.open(vou), lang="spa", config="--oem 1 --psm 6")
print(texto_reto)

In [ ]:
# Reto: la foto que leyó Tesseract (el texto de arriba salió de esta imagen)
ver([Image.open(vou)], ["VOU-0001.jpg: la foto tal como llega al OCR"], ancho=520, por_fila=1)

**Cómo leer esta salida**

- **Qué se ve:** La foto del comprobante tal como llega al OCR. El texto impreso en la celda anterior es lo que Tesseract entendió de esta imagen.
- **Qué significan los números:** Esta celda no tiene números: se compara a simple vista lo que se ve en la foto con el texto impreso arriba.
- **Qué se decide con esto:** Se anotan las causas que se ven (inclinación, sombra, baja resolución, perspectiva) en la celda siguiente; cada una tiene su tratamiento más adelante.

In [ ]:
# Causas visibles del mal resultado (se pueden editar)
causas = ["inclinación de la hoja", "sombra y fondo oscuro", "baja resolución (72 dpi)", "perspectiva de la foto"]
assert len(causas) >= 3, "Reto: se anotan al menos tres causas."
print("✔ Reto: causas anotadas ->", "; ".join(causas))

## Paso 2: PDF a imagen a 300 dpi (C1)

In [ ]:
# PDF a imagen con mi_pipeline: control C1
from mi_pipeline.ocr.renderizar import pdf_a_imagen

oficio = DATOS / "02_Oficios" / "OFI-0001.pdf"
imagen = pdf_a_imagen(oficio, dpi=300)
destino = SALIDAS / "OFI-0001_300dpi.png"
Image.fromarray(imagen).save(destino, dpi=(300, 300))
print("A 300 dpi:", imagen.shape, imagen.dtype, "| a 100 dpi:", pdf_a_imagen(oficio, dpi=100).shape)
# ✔ Control C1
assert destino.is_file(), "C1: no se guardó la imagen."
assert abs(imagen.shape[1] - 2480) <= 3 and abs(imagen.shape[0] - 3508) <= 3, "C1: una hoja A4 a 300 dpi mide unos 2480 x 3508 píxeles."
CONTROLES.append("C1")
print("✔ C1 superado: la imagen del PDF a 300 dpi existe ->", destino)

In [ ]:
# Paso 2, a la vista: la misma esquina del oficio a 100 dpi y a 300 dpi
pequena = pdf_a_imagen(oficio, dpi=100)
region_y, region_x = (0.045, 0.085), (0.25, 0.55)
ver([recortar(pequena, region_y, region_x), recortar(imagen, region_y, region_x)],
    ["100 dpi: la región tiene 3 veces menos puntos", "300 dpi: la misma región, más nítida"], ancho=620, por_fila=2, exacto=True)

**Cómo leer esta salida**

- **Qué se ve:** La misma región del encabezado del oficio a 100 dpi y a 300 dpi, agrandada al mismo tamaño en pantalla. A 100 dpi las letras se arman con pocos puntos y se ven con bordes escalonados; a 300 dpi se distinguen sus trazos.
- **Qué significan los números:** La salida de la celda anterior, `(3509, 2481, 3)`, es alto, ancho y tres colores, en píxeles. A 100 dpi la hoja mide un tercio por lado (unos 1170 x 827).
- **Qué se decide con esto:** Se dibuja a 300 dpi antes de leer: con menos puntos, Tesseract confunde letras parecidas.

## Paso 3: CER y recetas de limpieza de la imagen

In [ ]:
# CER del reto: cuánto del comprobante leyó mal Tesseract
from mi_pipeline.calidad.metricas import cer

correcto_vou = json.loads((DATOS / "10_Ground_Truth" / "VOU-0001.json").read_text(encoding="utf-8"))["texto_completo"]
cer_reto = cer(correcto_vou, texto_reto)
print(f"CER de la lectura cruda del comprobante: {cer_reto:.2f}")
assert cer_reto > 0.3, "El reto debía salir ilegible (CER mayor que 0,3)."

**Cómo leer esta salida**

- **Qué se ve:** Un solo número: el CER de la lectura cruda del comprobante.
- **Qué significan los números:** El CER es la proporción de caracteres que cambian entre el texto correcto y el leído: 0 es perfecto, 0,5 es que la mitad está mal y puede pasar de 1 si el OCR inventa texto.
- **Qué se decide con esto:** Es la línea base: toda mejora de la sesión se mide contra este número. Si una receta no lo baja, no sirve para ese documento.

## Paso 4: Primeras pruebas con pytest

In [ ]:
# ✔ Pruebas automáticas de los pasos 2 a 4
assert correr_pruebas(), "Alguna prueba falla: se lee el mensaje de pytest y se corrige el archivo que indica."
print("✔ Las pruebas de mi_pipeline pasan")

## Paso 5: Configuración en YAML

In [ ]:
# Los parámetros que lee mi_pipeline (archivo YAML)
import yaml

ruta_yaml = RAIZ / "mi_pipeline" / "config" / "pipeline.yaml"
parametros = yaml.safe_load(ruta_yaml.read_text(encoding="utf-8"))
print(json.dumps(parametros, ensure_ascii=False, indent=1))
assert parametros["ocr"]["dpi"] == 300 and parametros["ocr"]["lang"] == "spa"

**Cómo leer esta salida**

- **Qué se ve:** Los parámetros del programa en un diccionario: receta, resolución, idioma y umbral del puntaje, entre otros.
- **Qué significan los números:** `dpi: 300` es la resolución del paso 2; `umbral_score: 0.85` es el puntaje mínimo para aceptar una lectura sin revisión; `receta` es el tratamiento que se aplica por defecto.
- **Qué se decide con esto:** Si un número no sirve, se cambia en este archivo y no en el código; las demás celdas lo toman de aquí.

## Paso 6: `ejecutar` encadena los pasos

In [ ]:
# ejecutar: PDF -> imagen -> enderezar -> receta
from mi_pipeline.pipeline import cargar_configuracion, configurar_tesseract, ejecutar

CFG = cargar_configuracion()
print("Tesseract:", Path(configurar_tesseract(CFG)).name)
resultado = ejecutar(DATOS / "02_Oficios" / "OFI-0001.pdf", CFG, receta="crudo")
print(resultado["archivo"], "receta:", resultado["receta"], "inclinación:", resultado["inclinacion"], "grados", "imagen:", resultado["imagen"].shape)
assert abs(resultado["inclinacion"] - 4.98) < 0.5

**Cómo leer esta salida**

- **Qué se ve:** Una línea con el nombre del archivo, la receta, la inclinación medida y la forma de la imagen que devuelve `ejecutar`.
- **Qué significan los números:** `inclinación` está en grados (la real de este oficio es 4,98°); `imagen` es (alto, ancho, colores) en píxeles.
- **Qué se decide con esto:** Una sola función procesa cualquier documento con el mismo tratamiento. Si la inclinación sale muy distinta de la esperada, algo falló antes del OCR.

## E1: CER por receta (C2), ángulo (C3) y resolución

In [ ]:
# E1: CER por receta de limpieza de la imagen, control C2
from mi_pipeline.ocr.preprocesar import RECETAS, aplicar_receta, corregir_inclinacion

OCR = CFG["ocr"]
OFICIOS = ["OFI-0001", "OFI-0005", "OFI-0012", "OFI-0015", "OFI-0016"]
VOUCHERS = ["VOU-0001", "VOU-0002", "VOU-0003"]
ESTADOS = ["EFI-0031", "EFI-0035", "EFI-0037"]


def verdad(doc_id):
    """Respuesta esperada de un documento ficticio: texto correcto, campos y regiones."""
    return json.loads((DATOS / "10_Ground_Truth" / f"{doc_id}.json").read_text(encoding="utf-8"))


def cargar(doc_id, enderezar=True):
    """Imagen de un oficio (PDF a 300 dpi) o de un comprobante (foto), enderezada o no."""
    if doc_id.startswith("OFI"):
        imagen = pdf_a_imagen(DATOS / "02_Oficios" / f"{doc_id}.pdf", dpi=OCR["dpi"])
    else:
        imagen = np.array(Image.open(next((DATOS / "08_Vouchers").glob(f"{doc_id}.*"))).convert("RGB"))
    return corregir_inclinacion(imagen)[0] if enderezar else imagen


def leer_texto(imagen):
    """Texto que lee Tesseract con los parámetros del YAML."""
    return pytesseract.image_to_string(imagen, lang=OCR["lang"], config=f"--oem {OCR['oem']} --psm {OCR['psm']}")


cer_por_receta = {}
for doc_id in OFICIOS + VOUCHERS:
    imagen = cargar(doc_id, enderezar=doc_id.startswith("OFI"))
    correcto = verdad(doc_id)["texto_completo"]
    cer_por_receta[doc_id] = {receta: cer(correcto, leer_texto(aplicar_receta(imagen, receta))) for receta in RECETAS}
tabla_cer = pd.DataFrame(cer_por_receta).T.round(3)
tabla_cer["mejor"] = tabla_cer[list(RECETAS)].idxmin(axis=1)
print(tabla_cer)
oficios = tabla_cer.loc[OFICIOS]
print("\nCER medio de los 5 oficios:", oficios[list(RECETAS)].mean().round(3).to_dict())
# ✔ Control C2
assert oficios[list(RECETAS)].min(axis=1).mean() < oficios["crudo"].mean(), "C2: la mejor receta de cada oficio debe bajar el CER frente al crudo."
assert oficios[OCR["receta"]].mean() < oficios["crudo"].mean(), f"C2: la receta '{OCR['receta']}' del YAML no mejora al crudo."
CONTROLES.append("C2")
print(f"✔ C2 superado: CER medio de '{OCR['receta']}' = {oficios[OCR['receta']].mean():.3f} frente a {oficios['crudo'].mean():.3f} del crudo")

**Cómo leer esta salida**

- **Qué se ve:** Una tabla con un documento por fila y una receta por columna; la columna `mejor` nombra la receta con el CER más bajo. Después, el CER medio de los cinco oficios por receta.
- **Qué significan los números:** Cada celda es un CER: menor es mejor. Medias de los 5 oficios: crudo 0,268, gris 0,249, Otsu 0,43, adaptativa 0,236, mediana 0,147.
- **Qué se decide con esto:** Ninguna receta gana en todos los documentos: la sombra pide umbral adaptativo (OFI-0005 y OFI-0016) y el ruido de puntos sueltos pide mediana (OFI-0012 y OFI-0015). Se deja en el YAML la de menor CER medio.

In [ ]:
# E1, a la vista: cómo cambia el documento con cada receta (el CER de cada una va en el título)
def tira_de_recetas(doc_id, enderezar, franja):
    original = cargar(doc_id, enderezar=enderezar)
    imagenes = [recortar(aplicar_receta(original, receta), y=franja) for receta in RECETAS]
    return imagenes, [f"{receta}: CER {tabla_cer.loc[doc_id, receta]:.2f}" for receta in RECETAS]


imagenes, titulos = tira_de_recetas("OFI-0005", True, (0.25, 0.45))
ver(imagenes, [f"OFI-0005 {t}" for t in titulos], ancho=420, por_fila=3)
imagenes, titulos = tira_de_recetas("VOU-0001", False, (0.0, 1.0))
ver(imagenes, [f"VOU-0001 {t}" for t in titulos], ancho=300, por_fila=5)

**Cómo leer esta salida**

- **Qué se ve:** Arriba, un fragmento de OFI-0005 con las cinco recetas: `crudo` es la imagen sin tratar, `gris` quita el color, `otsu` pasa a blanco y negro con un solo umbral, `adaptativa` calcula el umbral por zona y `mediana` quita puntos sueltos. Abajo, el comprobante VOU-0001 con las mismas cinco.
- **Qué significan los números:** El título de cada imagen trae el CER de esa receta, el mismo de la tabla anterior. En OFI-0005, `otsu` deja casi negra la zona con sombra y el CER sube; `adaptativa` y `mediana` la limpian y el CER baja a cerca de 0,06. En el comprobante ocurre lo contrario: ninguna receta lo mejora y varias lo empeoran (el texto se rompe).
- **Qué se decide con esto:** Se elige la receta que baja el CER medio de los oficios. El comprobante no se arregla con una receta de limpieza porque su problema es otro: la foto tiene perspectiva y baja resolución; la resolución se ataca en la celda de más abajo.

In [ ]:
# E1: error del ángulo de enderezado, control C3
from mi_pipeline.ocr.preprocesar import a_gris, estimar_inclinacion

filas = []
for doc_id in OFICIOS + VOUCHERS:
    estimado = estimar_inclinacion(a_gris(cargar(doc_id, enderezar=False)))
    real = verdad(doc_id)["degradacion"]["rotacion"]
    filas.append({"documento": doc_id, "real": real, "estimado": estimado, "error": round(abs(estimado - real), 2)})
tabla_angulos = pd.DataFrame(filas).set_index("documento")
print(tabla_angulos)
# ✔ Control C3
assert tabla_angulos.loc[OFICIOS, "error"].max() < 0.5, "C3: el ángulo corregido de cada oficio debe diferir del real en menos de 0,5 grados."
CONTROLES.append("C3")
print(f"✔ C3 superado: error máximo en los oficios = {tabla_angulos.loc[OFICIOS, 'error'].max():.2f}°")

**Cómo leer esta salida**

- **Qué se ve:** Una tabla con el ángulo real (del ground truth) y el estimado de cada documento; `error` es la diferencia.
- **Qué significan los números:** Los ángulos están en grados. El error máximo en los oficios es 0,11° (el control exige menos de 0,5°). En los comprobantes el error es mayor (9,37° y 2,7°) porque son fotos con perspectiva y no solo están giradas.
- **Qué se decide con esto:** Los oficios se enderezan antes del OCR. Para los comprobantes el giro no basta.

In [ ]:
# E1, a la vista: el oficio inclinado y el mismo oficio enderezado (las líneas deben quedar horizontales)
inclinado, derecho = cargar("OFI-0001", enderezar=False), cargar("OFI-0001", enderezar=True)
giro = tabla_angulos.loc["OFI-0001"]
ver([recortar(inclinado, y=(0.0, 0.30)), recortar(derecho, y=(0.0, 0.30))],
    [f"Antes: inclinado {giro['real']}°", f"Después: girado {giro['estimado']}° en sentido contrario"], ancho=620, por_fila=2)

**Cómo leer esta salida**

- **Qué se ve:** El mismo oficio antes y después de enderezarlo: las líneas del membrete pasan de inclinadas a horizontales.
- **Qué significan los números:** El título de la imagen derecha dice cuántos grados se giró la hoja, en sentido contrario al estimado.
- **Qué se decide con esto:** Se endereza antes de leer porque Tesseract espera líneas horizontales; con un ángulo mal estimado el texto quedaría torcido.

In [ ]:
# E1: la resolución también cuenta: el comprobante ampliado
from mi_pipeline.ocr.preprocesar import ampliar

filas = {}
for doc_id in VOUCHERS:
    foto = cargar(doc_id, enderezar=False)
    correcto = verdad(doc_id)["texto_completo"]
    filas[doc_id] = {f"x{factor}": cer(correcto, leer_texto(ampliar(foto, factor) if factor > 1 else foto)) for factor in (1, 2, 3)}
tabla_ampliar = pd.DataFrame(filas).T.round(3)
print(tabla_ampliar)
assert tabla_ampliar["x3"].mean() < tabla_ampliar["x1"].mean(), "Ampliar la foto debía bajar el CER."

**Cómo leer esta salida**

- **Qué se ve:** Una tabla con el CER de cada comprobante a su tamaño (`x1`), agrandado 2 veces (`x2`) y 3 veces (`x3`).
- **Qué significan los números:** CER de `x1` a `x3`: VOU-0001 0,52 a 0,09; VOU-0002 0,417 a 0,28; VOU-0003 0,892 a 0,468.
- **Qué se decide con esto:** Si el CER baja al agrandar, la causa era la resolución y no la limpieza: se agranda la foto antes de leerla.

In [ ]:
# E1, a la vista: el comprobante a su tamaño y agrandado 3 veces
foto = cargar("VOU-0001", enderezar=False)
ver([recortar(foto, y=(0.22, 0.42), x=(0.18, 0.72)), recortar(ampliar(foto, 3), y=(0.22, 0.42), x=(0.18, 0.72))],
    [f"x1: CER {tabla_ampliar.loc['VOU-0001', 'x1']:.2f}", f"x3: CER {tabla_ampliar.loc['VOU-0001', 'x3']:.2f}"], ancho=620, por_fila=2, exacto=True)

**Cómo leer esta salida**

- **Qué se ve:** Las líneas de datos del comprobante a su tamaño (`x1`) y agrandadas 3 veces (`x3`), mostradas al mismo ancho, con el CER en el título. En `x1` las letras se ven con bordes escalonados; en `x3` hay más puntos por letra.
- **Qué significan los números:** Agrandar no inventa detalle, pero da a Tesseract letras con más puntos para leer; por eso el CER baja.
- **Qué se decide con esto:** Una foto de 72 dpi se agranda antes de pasarla al OCR.

## Paso 7: Confianza, validadores y puntaje (C4)

In [ ]:
# E2: validadores: el RUC y la fecha tienen reglas
from mi_pipeline.calidad.puntuar import puntuar, validar_fecha, validar_monto, validar_ruc

ruc = verdad("OFI-0001")["campos"]["ruc"]
ruc_malo = ruc[:-1] + str((int(ruc[-1]) + 1) % 10)
print("RUC del oficio:", ruc, "->", validar_ruc(ruc), "| con un dígito cambiado:", ruc_malo, "->", validar_ruc(ruc_malo))
print("Fechas:", validar_fecha("24 de junio de 2026"), validar_fecha("31 de junio de 2026"), "| montos:", validar_monto("S/ 1,250.00"), validar_monto("1,25.00"))
assert validar_ruc(ruc) and not validar_ruc(ruc_malo)

**Cómo leer esta salida**

- **Qué se ve:** Seis resultados `True` o `False`: el RUC correcto, el mismo RUC con un dígito cambiado, dos fechas y dos montos.
- **Qué significan los números:** `True` significa que el dato pasa su regla: dígito verificador del RUC, fecha que existe en el calendario, coma de miles bien puesta. El 31 de junio no existe y `1,25.00` tiene la coma mal puesta.
- **Qué se decide con esto:** Un RUC mal leído parece un número normal y solo la regla lo delata. Por eso el puntaje de la celda siguiente combina la confianza con estos validadores.

In [ ]:
# E2: el puntaje separa lecturas buenas de malas, control C4
from mi_pipeline.ocr.leer import leer_con_confianza
from mi_pipeline.pipeline import enrutar

filas = []
for doc_id in OFICIOS + VOUCHERS:
    imagen = cargar(doc_id, enderezar=doc_id.startswith("OFI"))
    correcto = verdad(doc_id)["texto_completo"]
    for receta in RECETAS:
        lectura = leer_con_confianza(aplicar_receta(imagen, receta), OCR["lang"], OCR["oem"], OCR["psm"])
        puntaje = puntuar(lectura["texto"], lectura["confianza"])
        filas.append({"documento": doc_id, "receta": receta, "cer": round(cer(correcto, lectura["texto"]), 3),
                      "confianza": puntaje["confianza"], "invalidos": puntaje["invalidos"], "puntaje": puntaje["score"],
                      "decision": enrutar(puntaje, CFG["calidad"]["umbral_score"])})
lecturas = pd.DataFrame(filas)
buenas, malas = lecturas[lecturas["cer"] < 0.15], lecturas[lecturas["cer"] > 0.30]
print(f"Lecturas buenas (CER < 0,15): {len(buenas)}, puntaje medio {buenas['puntaje'].mean():.2f}, mínimo {buenas['puntaje'].min():.2f}, aceptadas {(buenas['decision'] == 'aceptar').sum()}")
print(f"Lecturas malas  (CER > 0,30): {len(malas)}, puntaje medio {malas['puntaje'].mean():.2f}, máximo {malas['puntaje'].max():.2f}, aceptadas {(malas['decision'] == 'aceptar').sum()}")
print("\nLectura con CER bajo pero un campo inválido (el puntaje la manda a revisión):")
print(lecturas[(lecturas["invalidos"] > 0) & (lecturas["cer"] < 0.15)])
# ✔ Control C4
assert buenas["puntaje"].mean() - malas["puntaje"].mean() > 0.3, "C4: el puntaje medio de las lecturas buenas debe superar al de las malas por más de 0,3."
assert (malas["decision"] == "aceptar").mean() < 0.1, "C4: casi ninguna lectura mala debe aceptarse sin revisión."
assert (buenas["decision"] == "aceptar").mean() > 0.8, "C4: la mayoría de las lecturas buenas debe aceptarse."
CONTROLES.append("C4")
print("✔ C4 superado: el RUC válido pasa, el inválido falla y el puntaje separa las lecturas buenas de las malas")

**Cómo leer esta salida**

- **Qué se ve:** Dos resúmenes (lecturas buenas, con CER menor que 0,15, y lecturas malas, con CER mayor que 0,30) y una tabla con las lecturas de CER bajo que tienen un campo inválido.
- **Qué significan los números:** `confianza` es el promedio de lo seguro que está Tesseract (0 a 1); `invalidos` cuenta los campos que no pasan su validador; `puntaje` combina ambos a partes iguales; `decision` es aceptar, revisar o rechazar según el umbral 0,85. Las lecturas buenas promedian 0,93 y las malas 0,50.
- **Qué se decide con esto:** Las buenas deben aceptarse y las malas no. La tabla final muestra el caso clave: una lectura con CER bajo pero un campo inválido va a revisión aunque Tesseract parezca seguro.

In [ ]:
# E2, a la vista: la confianza de cada palabra en color (verde: segura; naranja: dudosa; rojo: insegura)
def pintar_confianza(doc_id):
    imagen = aplicar_receta(cargar(doc_id), OCR["receta"])
    lectura = leer_con_confianza(imagen, OCR["lang"], OCR["oem"], OCR["psm"])
    colores = [(0, 150, 0) if p["conf"] >= 80 else ((235, 130, 0) if p["conf"] >= 60 else (210, 0, 0)) for p in lectura["palabras"]]
    return con_cajas(imagen, [p["caja"] for p in lectura["palabras"]], colores, grosor=3), lectura["confianza"]


pintadas = {doc_id: pintar_confianza(doc_id) for doc_id in ("OFI-0001", "OFI-0012")}
ver([recortar(pagina, y=(0.20, 0.45)) for pagina, _ in pintadas.values()],
    [f"{doc_id}: confianza media {confianza:.2f}" for doc_id, (_, confianza) in pintadas.items()], ancho=620, por_fila=2)

**Cómo leer esta salida**

- **Qué se ve:** Un fragmento de dos oficios con una caja por palabra: verde si Tesseract está seguro (80 o más de 100), naranja si duda (60 a 80) y rojo si no (menos de 60).
- **Qué significan los números:** El título trae la confianza media de todo el oficio. OFI-0012 tiene más palabras en naranja y rojo que OFI-0001.
- **Qué se decide con esto:** El color indica dónde mirar: en un documento con muchas palabras rojas se revisa esa zona a mano.

In [ ]:
# ejecutar de punta a punta sobre los cinco oficios, con la receta del YAML
filas = []
for doc_id in OFICIOS:
    r = ejecutar(DATOS / "02_Oficios" / f"{doc_id}.pdf", CFG)
    filas.append({"documento": doc_id, "receta": r["receta"], "inclinación": r["inclinacion"], "confianza": round(r["confianza"], 3),
                  "puntaje": r["puntaje"]["score"], "decisión": r["decision"]})
print(pd.DataFrame(filas))
assert all(f["decisión"] in ("aceptar", "revisar", "rechazar") for f in filas)

**Cómo leer esta salida**

- **Qué se ve:** Una fila por oficio con la receta usada, la inclinación, la confianza, el puntaje y la decisión.
- **Qué significan los números:** `confianza` y `puntaje` van de 0 a 1; la decisión sale de comparar el puntaje con 0,85 y de los campos inválidos.
- **Qué se decide con esto:** Tres oficios se aceptan y dos pasan a revisión (OFI-0012 por baja confianza y OFI-0016 por un campo inválido). Es la tabla que se retoma en el caso en equipo.

## Lote público real (opcional): DIV-0006 y DIV-0007

In [ ]:
# Lote real (opcional): solo se mira el antes y el después, y se puntúa
import pymupdf
from IPython.display import display

REAL = DATOS / "14_Publico_Real"
if REAL.is_dir():
    filas = []
    for nombre in ("DIV-0006", "DIV-0007"):
        archivo = REAL / "01_Dividendos" / f"{nombre}.pdf"
        with pymupdf.open(str(archivo)) as pdf:
            paginas = len(pdf)
        for pagina in range(paginas):
            r = ejecutar(archivo, CFG, pagina=pagina)
            filas.append({"documento": nombre, "página": pagina + 1, "inclinación": r["inclinacion"], "confianza": round(r["confianza"], 3),
                          "campos válidos": r["puntaje"]["validos"], "inválidos": r["puntaje"]["invalidos"], "puntaje": r["puntaje"]["score"], "decisión": r["decision"]})
    print(pd.DataFrame(filas))
    antes = pdf_a_imagen(REAL / "01_Dividendos" / "DIV-0006.pdf", dpi=100)
    despues = aplicar_receta(corregir_inclinacion(antes)[0], OCR["receta"])
    display(Image.fromarray(np.hstack([a_gris(antes), despues])))
else:
    print("Sección opcional omitida: en esta máquina no hay carpeta '14_Publico_Real'.")

**Cómo leer esta salida**

- **Qué se ve:** Una fila por página de dos documentos reales de la SMV y, debajo, una página en gris y después de la limpieza. Si aparece el aviso de sección omitida, esta parte no aplica en esta máquina.
- **Qué significan los números:** No hay texto correcto de referencia, así que no hay CER: solo `confianza`, campos válidos e inválidos y `puntaje`. La imagen se muestra a 100 dpi solo para verla en pantalla: a esa resolución la limpieza deja letras rotas, y por eso `ejecutar` procesa a 300 dpi.
- **Qué se decide con esto:** Con documentos reales se decide con el puntaje y los validadores; las páginas con campos inválidos van a revisión humana.

## Paso 8: E3, IoU de firma y sello (C5) y zonas de la página (C5b)

In [ ]:
# E3: IoU de la firma y el sello detectados con OpenCV (salida calculada de antemano), control C5
from mi_pipeline.calidad.metricas import iou

caja_a, caja_b = (0, 0, 10, 10), (5, 0, 15, 10)
print("IoU a mano: zona común 50 / zona total 150 =", round(50 / 150, 3), "| con la función:", round(iou(caja_a, caja_b), 3))
assert abs(iou(caja_a, caja_b) - 50 / 150) < 1e-9

filas = []
for doc_id in OFICIOS:
    detectado = json.loads((DATOS / "11_Salidas_Precalculadas" / "ocr" / f"firma_sello_{doc_id}.json").read_text(encoding="utf-8"))
    correcto = {region["tipo"]: region["bbox"] for region in verdad(doc_id)["regiones"]}
    for region in detectado["regiones"]:
        filas.append({"documento": doc_id, "región": region["tipo"], "iou": round(iou(region["bbox"], correcto[region["tipo"]]), 3)})
tabla_iou = pd.DataFrame(filas).pivot(index="documento", columns="región", values="iou")
print(tabla_iou)
# ✔ Control C5
assert tabla_iou["sello"].min() > 0.85, "C5: el sello debe superar 0,85 de IoU en los cinco oficios."
assert (tabla_iou["firma"] >= 0.5).sum() >= 4, "C5: la firma debe llegar a 0,5 de IoU en al menos cuatro oficios."
CONTROLES.append("C5")
print(f"✔ C5 superado: IoU del sello {tabla_iou['sello'].min():.2f} a {tabla_iou['sello'].max():.2f}; la firma llega a 0,5 en {(tabla_iou['firma'] >= 0.5).sum()} de 5")

**Cómo leer esta salida**

- **Qué se ve:** Primero un IoU calculado a mano y con la función; después una tabla de IoU por oficio para el sello y la firma.
- **Qué significan los números:** El IoU va de 0 a 1: 1 es que la caja detectada coincide con la correcta y 0 que no se tocan. Sello: 0,901 a 0,945; firma: 0,043 a 0,677.
- **Qué se decide con esto:** El sello se detecta bien en los cinco oficios. La firma falla en OFI-0015, donde solo se halló la raya inferior (IoU 0,043); ese oficio se revisa.

In [ ]:
# E3, a la vista: caja correcta (verde) y caja detectada (roja) sobre el oficio
def oficio_con_cajas(doc_id):
    detectado = json.loads((DATOS / "11_Salidas_Precalculadas" / "ocr" / f"firma_sello_{doc_id}.json").read_text(encoding="utf-8"))
    correcto = {region["tipo"]: region["bbox"] for region in verdad(doc_id)["regiones"]}
    pagina = a_pil(cargar(doc_id)).resize((detectado["ancho_px"], detectado["alto_px"]))
    cajas, colores, valores = [], [], {}
    for region in detectado["regiones"]:
        cajas += [correcto[region["tipo"]], region["bbox"]]
        colores += [(0, 150, 0), (210, 0, 0)]
        valores[region["tipo"]] = iou(region["bbox"], correcto[region["tipo"]])
    return recortar(con_cajas(pagina, cajas, colores, grosor=6), y=(0.45, 0.70)), valores


imagenes, titulos = [], []
for doc_id in ("OFI-0001", "OFI-0015"):
    imagen_doc, valores = oficio_con_cajas(doc_id)
    imagenes.append(imagen_doc)
    titulos.append(f"{doc_id}: IoU sello {valores['sello']:.2f}, firma {valores['firma']:.2f}")
ver(imagenes, titulos, ancho=620, por_fila=2)

**Cómo leer esta salida**

- **Qué se ve:** Dos oficios con la caja correcta (verde) y la detectada (roja) de sello y firma.
- **Qué significan los números:** El título repite el IoU: cuanto más se superponen la caja verde y la roja, más cerca de 1.
- **Qué se decide con esto:** En OFI-0015 la caja roja de la firma no cubre la verde: es lo que el IoU bajo expresa en número.

In [ ]:
# E3: encabezado y pie por coordenadas, control C5b
from mi_pipeline.ocr.zonas import ZONAS, recortar_zona, zonas_de_pagina

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "setiembre", "octubre", "noviembre", "diciembre"]


def normalizar(texto):
    """Minúsculas y espacios simples, para buscar un campo dentro de un texto leído."""
    return " ".join(texto.lower().split())


print("Zonas (fracciones de la altura):", {nombre: f"{desde:.2f} a {hasta:.2f}" for nombre, (desde, hasta) in ZONAS.items()})
filas = []
for doc_id in OFICIOS:
    pagina = aplicar_receta(cargar(doc_id), OCR["receta"])
    zonas = zonas_de_pagina(pagina)
    assert recortar_zona(pagina, *ZONAS["pie"]).shape == zonas["pie"].shape
    encabezado = normalizar(leer_texto(zonas["encabezado"]))
    pie = leer_con_confianza(zonas["pie"], OCR["lang"], OCR["oem"], OCR["psm"])
    campos = verdad(doc_id)["campos"]
    anio, mes, dia = campos["fecha"].split("-")
    filas.append({"documento": doc_id,
                  "emisor": normalizar(campos["emisor"]) in encabezado,
                  "número de oficio": "-".join(campos["numero_oficio"].split("-")[:2]) in encabezado,
                  "fecha": f"{int(dia)} de {MESES[int(mes) - 1]} de {anio}" in encabezado,
                  "palabras seguras en el pie": len([p for p in pie["palabras"] if p["conf"] >= 60 and len(p["texto"]) >= 3])})
tabla_zonas = pd.DataFrame(filas).set_index("documento")
print(tabla_zonas)
# ✔ Control C5b
assert tabla_zonas["emisor"].all(), "C5b: el emisor debe aparecer en el encabezado de los cinco oficios."
assert tabla_zonas["número de oficio"].sum() >= 4 and tabla_zonas["fecha"].sum() >= 4, "C5b: el número de oficio y la fecha deben aparecer en el encabezado de al menos cuatro oficios."
assert (tabla_zonas["palabras seguras en el pie"] == 0).all(), "C5b: el pie de estos oficios está en blanco."
CONTROLES.append("C5b")
print("✔ C5b superado: el encabezado trae emisor, número y fecha; el pie está en blanco")

**Cómo leer esta salida**

- **Qué se ve:** Las fracciones de la altura de cada zona y una tabla por oficio: `True` o `False` si el encabezado trae el emisor, el número de oficio y la fecha, y cuántas palabras seguras tiene el pie.
- **Qué significan los números:** Zonas: encabezado de 0 a 0,22 de la altura, cuerpo de 0,22 a 0,88 y pie de 0,88 a 1. El pie da 0 palabras seguras porque está en blanco en estos oficios. El número de oficio falla en OFI-0012, que tiene ruido de puntos sueltos.
- **Qué se decide con esto:** Con solo coordenadas se separa el membrete del cuerpo y se busca cada campo en su zona; un campo que falla se revisa en el documento y no se da por bueno.

In [ ]:
# E3, a la vista: las tres zonas de la página y el encabezado recortado
pagina = aplicar_receta(cargar("OFI-0001"), OCR["receta"])
alto, ancho_pagina = pagina.shape[:2]
colores_zonas = {"encabezado": (0, 90, 200), "cuerpo": (0, 150, 0), "pie": (210, 0, 0)}
marcada = con_cajas(pagina, [(0, int(alto * d), ancho_pagina - 1, int(alto * h) - 1) for d, h in ZONAS.values()], list(colores_zonas.values()), grosor=14)
ver([marcada], ["Zonas: azul encabezado, verde cuerpo, rojo pie"], ancho=520, por_fila=1)
ver([recortar_zona(pagina, *ZONAS["encabezado"])], ["El encabezado recortado: de aquí salen emisor, número y fecha"], ancho=700, por_fila=1)

**Cómo leer esta salida**

- **Qué se ve:** El oficio con las tres zonas marcadas por color y, debajo, el encabezado recortado.
- **Qué significan los números:** Los bordes marcan las fracciones 0 a 0,22, 0,22 a 0,88 y 0,88 a 1 de la altura de la hoja.
- **Qué se decide con esto:** Se recorta cada zona y se lee solo la que interesa: emisor, número y fecha salen del encabezado.

## Paso 9: E4: tabla de un estado financiero escaneado a DataFrame (C6)

In [ ]:
# E4: tabla del balance de un estado financiero escaneado a DataFrame, control C6
from mi_pipeline.ocr.tablas import cuadratura, tabla_a_dataframe

filas, tablas = [], {}
for doc_id in ESTADOS:
    pagina = corregir_inclinacion(pdf_a_imagen(DATOS / "03_Estados_Financieros" / f"{doc_id}.pdf", dpi=OCR["dpi"]))[0]
    tablas[doc_id] = tabla_a_dataframe(pagina, OCR["lang"], OCR["oem"], OCR["psm"])
    c = cuadratura(tablas[doc_id], "2026")
    filas.append({"documento": doc_id, "filas": len(tablas[doc_id]), "total activo leído": c["activo"],
                  "total activo correcto": verdad(doc_id)["campos"]["total_activo"], "diferencia": c["diferencia"], "cuadra": c["cuadra"]})
print(tablas["EFI-0031"])
print()
print(pd.DataFrame(filas))
# ✔ Control C6
assert all(f["cuadra"] for f in filas), "C6: Activo = Pasivo + Patrimonio debe cumplirse en los tres estados financieros."
assert all(f["total activo leído"] == f["total activo correcto"] for f in filas), "C6: el total de activo leído debe coincidir con el correcto."
CONTROLES.append("C6")
print("✔ C6 superado: Activo = Pasivo + Patrimonio en las tres tablas extraídas")

**Cómo leer esta salida**

- **Qué se ve:** Primero las filas de la tabla de EFI-0031 convertidas en DataFrame (concepto y cifras) y después un resumen de los tres estados financieros.
- **Qué significan los números:** `total activo leído` y `total activo correcto` deben coincidir; `diferencia` es Activo menos (Pasivo + Patrimonio); `cuadra` es `True` si la diferencia es cero. Los tres cuadran (activo de EFI-0031: 35.569).
- **Qué se decide con esto:** Si el balance cuadra y el total coincide, la tabla se acepta; si no cuadra, la lectura no es confiable y se revisa.

In [ ]:
# E4, a la vista: la tabla del balance antes y después de borrar sus líneas
from mi_pipeline.ocr.tablas import quitar_lineas

pagina = corregir_inclinacion(pdf_a_imagen(DATOS / "03_Estados_Financieros" / "EFI-0031.pdf", dpi=OCR["dpi"]))[0]
sin_lineas, cuantas = quitar_lineas(a_gris(pagina))
ver([recortar(pagina, y=(0.18, 0.50)), recortar(sin_lineas, y=(0.18, 0.50))],
    ["Original: la tabla con sus líneas", f"Sin líneas ({cuantas} detectadas): lo que lee Tesseract"], ancho=620, por_fila=2)
print(tablas["EFI-0031"].head(8))

**Cómo leer esta salida**

- **Qué se ve:** La tabla del balance con sus líneas, al lado la misma tabla sin ellas y, debajo, las primeras filas del DataFrame.
- **Qué significan los números:** El título de la imagen derecha dice cuántas líneas se detectaron y se borraron.
- **Qué se decide con esto:** Sin las líneas, Tesseract no las lee como letras; con el DataFrame, las cifras se pueden sumar y comprobar.

In [ ]:
# E4: Tesseract frente a PaddleOCR (resultados guardados de cinco oficios y una tabla; la celda siguiente ejecuta PaddleOCR en vivo)
precalculadas = DATOS / "11_Salidas_Precalculadas" / "ocr"
filas = []
for doc_id in OFICIOS:
    archivo = precalculadas / f"paddle_texto_{doc_id}.json"
    if not archivo.is_file():
        continue
    paddle = json.loads(archivo.read_text(encoding="utf-8"))
    correcto = verdad(doc_id)["texto_completo"]
    tesseract = ejecutar(DATOS / "02_Oficios" / f"{doc_id}.pdf", CFG)
    filas.append({"documento": doc_id, "CER Tesseract": round(cer(correcto, tesseract["texto"]), 3),
                  "CER PaddleOCR": round(cer(correcto, paddle["texto"]), 3), "segundos de PaddleOCR (CPU)": paddle["segundos_prediccion"]})
if filas:
    comparacion = pd.DataFrame(filas).set_index("documento")
    print(comparacion)
    print("\nCER medio:", comparacion[["CER Tesseract", "CER PaddleOCR"]].mean().round(3).to_dict())
else:
    print("No hay salidas calculadas de antemano de PaddleOCR en esta máquina.")
tabla_paddle = precalculadas / "paddle_tabla_EFI-0031.json"
if tabla_paddle.is_file():
    resultado_paddle = json.loads(tabla_paddle.read_text(encoding="utf-8"))
    filas_html = [re.findall(r"<td[^>]*>(.*?)</td>", fila, flags=re.S) for fila in re.findall(r"<tr>(.*?)</tr>", resultado_paddle["tablas"][0]["html"], flags=re.S)]
    print(f"\nPaddleOCR (tabla de EFI-0031): {len(filas_html)} filas leídas en {resultado_paddle['segundos_prediccion'] / 60:.0f} minutos de CPU "
          f"| Tesseract + OpenCV: {len(tablas['EFI-0031']) + 1} filas (con encabezado) en pocos segundos")
    print([fila for fila in filas_html if fila and fila[0].lower().startswith("total")])

**Cómo leer esta salida**

- **Qué se ve:** Una tabla con el CER de Tesseract y de PaddleOCR por oficio y el tiempo de PaddleOCR, y una comparación de las tablas de EFI-0031.
- **Qué significan los números:** PaddleOCR da un CER cercano a 0,03 frente a 0,15 de Tesseract con la receta por defecto, pero tarda unos 21 segundos por página frente a 1 o 2 de Tesseract.
- **Qué se decide con esto:** Es una decisión de precisión frente a costo. La tabla de PaddleOCR tardó 38 minutos en una CPU de 24 hilos, por eso no se ejecuta en clase.

In [ ]:
# E4: PaddleOCR en vivo sobre un oficio (CPU, modelos mobile; unos 30 segundos; la primera vez descarga 13 MB)
doc_id = "OFI-0001"
try:
    from paddleocr import PaddleOCR

    inicio = time.perf_counter()
    motor = PaddleOCR(text_detection_model_name="PP-OCRv5_mobile_det", text_recognition_model_name="latin_PP-OCRv5_mobile_rec",
                      use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
                      text_det_limit_side_len=2600, text_det_limit_type="max", enable_mkldnn=False)
    carga = time.perf_counter() - inicio
    pagina = corregir_inclinacion(pdf_a_imagen(DATOS / "02_Oficios" / f"{doc_id}.pdf", dpi=OCR["dpi"]))[0]
    inicio = time.perf_counter()
    lectura = motor.predict(np.ascontiguousarray(pagina[:, :, ::-1]))[0]
    segundos = time.perf_counter() - inicio
    texto_paddle = "\n".join(lectura["rec_texts"])
    print(f"Carga: {carga:.1f} s | lectura: {segundos:.1f} s | líneas: {len(lectura['rec_texts'])}")
    print("CER de PaddleOCR en vivo:", round(cer(verdad(doc_id)["texto_completo"], texto_paddle), 3))
    print(texto_paddle[:300])
except Exception as error:
    print("PaddleOCR no pudo ejecutarse en esta máquina:", error)
    print("La comparación de la celda anterior sigue siendo válida.")

**Cómo leer esta salida**

- **Qué se ve:** El tiempo de carga y de lectura, el número de líneas leídas, el CER de PaddleOCR y los primeros 300 caracteres del texto. Pueden aparecer avisos como `No ccache found` o `TqdmWarning`: no impiden la lectura.
- **Qué significan los números:** El CER de PaddleOCR sobre OFI-0001 se compara con el de Tesseract en ese oficio, que ronda 0,07; menor es mejor.
- **Qué se decide con esto:** PaddleOCR lee mejor este oficio pero tarda más. Si la celda avisa que no pudo ejecutarse, la comparación de la celda anterior sigue siendo válida.

## Lote público real (opcional): MEM-0002, página 47

In [ ]:
# Lote real (opcional): página 47 de una memoria anual: la capa de texto del PDF es la referencia
if REAL.is_dir():
    memoria = REAL / "02_Memorias" / "MEM-0002.pdf"
    with pymupdf.open(str(memoria)) as pdf:
        referencia = pdf[46].get_text()
    r = ejecutar(memoria, CFG, pagina=46)
    print(f"CER frente a la capa de texto: {cer(referencia, r['texto']):.3f}, puntaje {r['puntaje']['score']}, decisión {r['decision']}")
    tabla_memoria = tabla_a_dataframe(corregir_inclinacion(pdf_a_imagen(memoria, pagina=46, dpi=OCR["dpi"]))[0], OCR["lang"], OCR["oem"], OCR["psm"])
    total = tabla_memoria[tabla_memoria["Concepto"].str.lower().str.startswith("total activo")]
    print(total)
    assert "72,952,708" in referencia and 72952708.0 in total.iloc[0, 1:].tolist(), "El total de activo leído debe coincidir con la capa de texto."
else:
    print("Sección opcional omitida: en esta máquina no hay carpeta '14_Publico_Real'.")

**Cómo leer esta salida**

- **Qué se ve:** El CER frente a la capa de texto del PDF, el puntaje, la decisión y la fila de «total activo» de la tabla leída por OCR.
- **Qué significan los números:** Aquí sí hay referencia (el propio PDF trae el texto), por eso se calcula el CER: 0,009. El total de activo leído coincide con el del PDF.
- **Qué se decide con esto:** Es la lectura más fácil posible, porque el texto digital se convirtió en imagen: no se extrapola a documentos escaneados reales.

## Reto extra (opcional)

In [ ]:
# Reto extra (opcional): ¿psm 6 o psm 3 con la receta del YAML?
filas = {}
for doc_id in OFICIOS:
    imagen = aplicar_receta(cargar(doc_id), OCR["receta"])
    correcto = verdad(doc_id)["texto_completo"]
    filas[doc_id] = {f"psm {psm}": cer(correcto, pytesseract.image_to_string(imagen, lang=OCR["lang"], config=f"--oem {OCR['oem']} --psm {psm}")) for psm in (6, OCR["psm_pagina"])}
tabla_psm = pd.DataFrame(filas).T.round(3)
print(tabla_psm)
print("CER medio:", tabla_psm.mean().round(3).to_dict())

**Cómo leer esta salida**

- **Qué se ve:** Una tabla con el CER de cada oficio con `psm 6` (un bloque de texto) y con `psm 3` (página con varias zonas).
- **Qué significan los números:** CER medio: `psm 6` = 0,147, `psm 3` = 0,108; menor es mejor.
- **Qué se decide con esto:** Un parámetro se cambia solo si la medición lo respalda: en 25 oficios la diferencia casi desaparece, por eso el YAML conserva `psm: 6`.

## Cierre

In [ ]:
# ✔ Cierre: los controles y el tiempo del cuaderno
esperados = ["C0", "C1", "C2", "C3", "C4", "C5", "C5b", "C6"]
faltan = [c for c in esperados if c not in CONTROLES]
assert not faltan, f"Faltan controles por superar: {faltan}"
print("✔ Controles superados:", ", ".join(CONTROLES))
print(f"Tiempo del cuaderno: {(time.perf_counter() - INICIO) / 60:.1f} minutos")
print("Siguiente: guardar el avance (commit) y subir la rama, como indica la guía.")